<a href="https://colab.research.google.com/github/jeyner99/Integraci-n-de-datos-y-prospectiva/blob/main/Reto_2_Integraci%C3%B3n_de_datos_Brainer_Arango_y_Jeyner_Casta%C3%B1o.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

#Base de datos:
La base de datos utilizada recopila las veces que se materializaron riesgos en una FINTECH, esta contiene variables como la frecuencia, variable mide cantidad de veces que se materializó un riesgo en un periodo de tiempo y además la base de datos contiene la variable severidad, esta variable registra la cantidad de dinero que se gastó debido a la materialización de un riesgo.

La variable frecuencia y severidad permiten calcular la distribución agregada de las pérdidas, variable utilizada para medir las pérdidas ocasionadas por la materialización de riesgos en las organizaciones, esta variable es clave a la hora de tomar decisiones para el tratamiento del riesgo y para la asegurabilidad de riesgos.

El $muestreo$ es el primer metodo de integración de datos (Por clusterización). La idea es crear nuevos datos  a partir de los que tengo para aumntar confiabilidad

Se rige por el indice de confiabilidad. Tener mil datos como minimo para una confiablidad del 99,9% (Estandar internacional)

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
nxl="/content/drive/MyDrive/Universidad/2026-2/Integración de datos y prospectiva/Datos_Riesgo_Operacional.xlsx"
XDB=pd.read_excel(nxl)
XDB.head()
Freq=XDB.iloc[:,3]; Sev=XDB.iloc[:,4]
LDA=Freq*Sev

plt.figure()
sns.histplot(LDA, color="blue", bins=10, kde = True)
plt.title("Distribución agregada de las pérdidas")
plt.grid()
plt.show()

plt.figure()
sns.boxplot(x=LDA, color="blue")
plt.title("Boxplot de las pérdidas agregadas")
plt.grid()
plt.show()

Los gráficos de cómo se distribuye LDA demuestra una gran concentración de datos en pérdias bajas, además muestra unos pocos datos que son muy altos, estos datos se podrían eliminar si hacemos eliminación de outliers.

Es importante visualizar los gráficos de los datos para hacernos una idea del tratamiento que debemos seguir a continuación para los futuros pasos que nos solicite nuestra organización.


##Medidas de tendencia central previas al muestreo aleatorio con el método de Montecarlo

In [ ]:
NI = 10 #Número de intervalos en los cuales quiero agrupar los datos
counts, Limits = np.histogram(LDA, bins = NI) #Agrupar los datos en 10 intervalos
LI = Limits[:-1] #Limites inferiores del histograma
LS = Limits[1:]  #Limites superiores del histograma
fi = counts/sum(counts) #Porcentaje de datos por intervalo
MC = (LI+LS)/2 #Marca de clase: Dato representativo de cada intervalo

df = pd.DataFrame(np.column_stack((LI, LS, counts, fi, MC)))
df.columns = ["LI", "LS", "ND", "fi", "Marca de clase"]
display(df)

In [ ]:
#Media - Valor esperado de una pérdida (Qué es lo más común que pase)
u = np.sum(MC*fi)
print("La media de las pérdidas es:", u)

#Varianza - Indica que tan dispersos están los datos alrededor de la media
varc = np.sum(((MC - u)**2)*fi)
print("La varianza de las pérdidas es:", varc)

#Desviación estándar
sigma = np.sqrt(varc)
print("La desviación estándar de las pérdidas es:", sigma)

#Intervalo de variación de las pérdidas
#En este intervalo se encuentra el 95% de los datos (2*sigma)
print("El limite inferior es ", u-2*sigma)
print("El limite superior es ", u+2*sigma)

#Coeficiente de asimetría
coef_asm = np.sum((((MC-u)**3)*fi)/(sigma**3))
print("El coeficiente de asimetría es:", coef_asm)

#Coeficiente de curtosis
curtosis = np.sum((((MC-u)**4*fi))/(sigma**4))-3
coef_curt = np.sum((((MC-u)**4)*fi)/((sigma**4)))
print("El coeficiente de curtosis es:", curtosis)

Lo primero que podemos notar es una asimetría muy alta, lo cual nos demuestra una gran concentración de los datos hacia la izquierda, esto en sintonía con el supuesto de que en la distribución agregada de las pérdidas por lo general los datos deben estar concentrados hacia este lado representando que la mayoría de las pérdidas deben ser menores para que sea rentable asegurar este tipo de negocios, porque si la mayoría de las pérdidas son catastroficas nunca va a ser rentable asegurar. Además podemos observar una curtosis cercana a 0 pero no igual a 0, esto demuestra concentración de datos considerable en intervalos cercanos a la media.

Estas medidas van a ser comparadas con el cambio que genera crear datos aleatorios para mejorar la confianza.

#2. Se procede con la creación de lo concentradores de información

In [ ]:
import random #Cogemos cinco datos random de los que tenemos
random.seed(42)
XC=np.array(random.choices(LDA, k=5))
XC=np.sort(XC)
print("Los concentradores de información son:", XC)

In [ ]:
Box0=[]; Box1=[];Box2=[];Box3=[];Box4=[]
Boxes=[Box0,Box1,Box2,Box3,Box4]
for k in range(len(LDA)):
  d=np.abs(XC-LDA.iloc[k,])/XC #La distancia de un dato a un concentrador
  nc=np.argmin(d)#Numero de la caja donde irá el dato
  #print("La distancia porcentual es de:", d)
  #print("El dato pertenece a la categoria:", nc)
  Boxes[nc].append(LDA.iloc[k,])

# Determinamos el número de datos a muestrear por caja una vez que todas las Boxes están llenas
pm = [] # Vector para almacenar el número de muestras por caja
for j in range(5):
    # Calculamos la proporción de datos en la caja actual
    proportion = len(Boxes[j]) / len(LDA)
    # Escalar esta proporción por la longitud total de LDA para obtener el número de muestras.
    # Usamos round para manejar posibles problemas de punto flotante y lo convertimos a entero.
    num_samples = int(round(proportion * len(LDA)))
    pm.append(num_samples)
pm = np.array(pm) # Convertir a un array de numpy
print("El número de datos a muestrear de cada caja es:", pm)

In [ ]:
LDA2=[]

for j in range(5):
    if len(Boxes[j]) > 0:
        LDA2.extend(random.choices(Boxes[j], k=pm[j]))

LDA2=np.array(LDA2)

#Se comprueban las metricas estadisticas
uo=np.mean(LDA)
print("La media de los datos observados es:", uo)
ue=np.mean(LDA2)
print("La media de los datos externos es:", ue)
du=np.abs((uo-ue)/uo)
print("El error relativo porcentual es:", du*100) # Corregido: multiplicado por 100 para porcentaje

plt.figure()
sns.kdeplot(LDA, color="red", label="Datos observados")
sns.kdeplot(LDA2, color="blue", label="Datos externos")
plt.grid()
plt.show()

**Analisis de resultado**
De acuerdo con el metodo de integración de datos por muestreo aleatorio y concentradores de información, la discrepancia que se obtuvo para la media de las distribuciones (Datos observados: LDA, Datos externos:LDA2) estuvo por debajo del 5% en promedio, lo que nos indica que el 95% de los datos aproximadamente fueron correctamente seleccionados. ESte proceso se hizo con el fin de lograr una confiabilidad del 99,9% en la estimacion de las perdidas.
**Calcular medidas anteriormente vistas en clase**

#Nuevas medidas de tendencia central de la Distribución agregada de pérdidas:

In [ ]:
NI = 10 #Número de intervalos en los cuales quiero agrupar los datos
counts, Limits = np.histogram(LDA2, bins = NI) #Agrupar los datos en 10 intervalos
LI = Limits[:-1] #Limites inferiores del histograma
LS = Limits[1:]  #Limites superiores del histograma
fi = counts/sum(counts) #Porcentaje de datos por intervalo
MC = (LI+LS)/2 #Marca de clase: Dato representativo de cada intervalo

df = pd.DataFrame(np.column_stack((LI, LS, counts, fi, MC)))
df.columns = ["LI", "LS", "ND", "fi", "Marca de clase"]
display(df)

In [ ]:
#Media - Valor esperado de una pérdida (Qué es lo más común que pase)
u = np.sum(MC*fi)
print("La media de las pérdidas es:", u)

#Varianza - Indica que tan dispersos están los datos alrededor de la media
varc = np.sum(((MC - u)**2)*fi)
print("La varianza de las pérdidas es:", varc)

#Desviación estándar
sigma = np.sqrt(varc)
print("La desviación estándar de las pérdidas es:", sigma)

#Intervalo de variación de las pérdidas
#En este intervalo se encuentra el 95% de los datos (2*sigma)
print("El limite inferior es ", u-2*sigma)
print("El limite superior es ", u+2*sigma)

#Coeficiente de asimetría
coef_asm = np.sum((((MC-u)**3)*fi)/(sigma**3))
print("El coeficiente de asimetría es:", coef_asm)

#Coeficiente de curtosis
curtosis = np.sum((((MC-u)**4*fi))/(sigma**4))-3
coef_curt = np.sum((((MC-u)**4)*fi)/((sigma**4)))
print("El coeficiente de curtosis es:", curtosis)

#Comparación de los cambios en las medidas de tendencia central después de realizar el muestreo:

In [ ]:
from scipy import stats
# Medidas estadísticas para LDA (Original)
print("--- Medidas Estadísticas para LDA (Original) ---")
print(f"Media de LDA: {np.mean(LDA)}")
print(f"Mediana de LDA: {np.median(LDA)}")
mode_lda_result = stats.mode(LDA, keepdims=False)
print(f"Moda de LDA: {mode_lda_result.mode} (con conteo de {mode_lda_result.count})")
print(f"Desviación Estándar de LDA: {np.std(LDA)}")
print(f"Asimetría de LDA: {stats.skew(LDA)}")
print(f"Curtosis de LDA: {stats.kurtosis(LDA)}")
print("\n")

# Medidas estadísticas para LDA2 (Después del muestreo)
print("--- Medidas Estadísticas para LDA2 (Después del muestreo) ---")
print(f"Media de LDA2: {np.mean(LDA2)}")
print(f"Mediana de LDA2: {np.median(LDA2)}")
mode_lda2_result = stats.mode(LDA2, keepdims=False)
print(f"Moda de LDA2: {mode_lda2_result.mode} (con conteo de {mode_lda2_result.count})")
print(f"Desviación Estándar de LDA2: {np.std(LDA2)}")
print(f"Asimetría de LDA2: {stats.skew(LDA2)}")
print(f"Curtosis de LDA2: {stats.kurtosis(LDA2)}")

Lo que podemos observar es que los cambios en las medidas de tendencia central realmente fueron muy ligeros, esto es un buen indicador debido a que lo que buscamos generando nuevos datos aleatorios es mejorar la confianza de nuestros datos llevandolos a mil datos, pero es importante que no se distorsione la realidad de los datos, eso lo podemos observar en las gráficas de cómo se distribuyen los datos antes y después:

In [ ]:

plt.figure()
sns.histplot(LDA, color="blue", bins=10, kde = True)
plt.title("Distribución agregada de las pérdidas")
plt.grid()
plt.show()

plt.figure()
sns.histplot(LDA2, color="red", bins=10, kde = True)
plt.title("Distribución agregada de las pérdidas después del muestreo")
plt.grid()
plt.show()

Como se observa en las gráficas la distribución agregada de las pérdidas no tuvo cambios significativos y siguen siendo una representación adecuada de los datos originales pero con una confianza mucho más alta.